In [ ]:
%%capture
# Installs Unsloth, Xformers (Flash Attention) and all other packages!
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# We have to check which Torch version for Xformers (2.3 -> 0.0.27)
from torch import __version__; from packaging.version import Version as V
xformers = "xformers==0.0.27" if V(__version__) < V("2.4.0") else "xformers"
!pip install --no-deps {xformers} trl peft accelerate bitsandbytes triton
!pip install Pillow

!pip install sentence_transformers
!pip install wandb
!pip install codebleu
!wandb login 511e83bfe9b483a47a3b5c7a55014647ffca0e75

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import requests
import json
import csv
import pandas as pd
import numpy as np
import os
import wandb
import torch
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.font_manager as fm
import matplotlib.image as mpimg
from io import StringIO, BytesIO
from tqdm import tqdm
import PIL
import gc

from datasets import load_dataset, Dataset
from codebleu import calc_codebleu

import warnings
warnings.filterwarnings("ignore")

folder_path = '/content/drive/My Drive/ChoAI'
os.chdir(folder_path)

# url = 'manual_add.json'
# response = requests.get(url)

os.environ["WANDB_PROJECT"] = "ChoAI"
os.environ["WANDB_LOG_MODEL"] = "checkpoint"

# logo_banner = 'horizontal-logo.png'
# logo_response = requests.get(logo_banner)
logo_img = mpimg.imread('horizontal-logo.png')


with open('final_file.json', 'r') as json_file:
    data = json.load(json_file)

# Load the CSV string into a pandas DataFrame
df = pd.read_csv('ProcessedData/Joined_DF.csv')
df['action_time'] = pd.to_datetime(df['action_time'],format='ISO8601')
df['Date']=pd.to_datetime(df['Date'])
manjari_bold_path = 'Manjari-Bold.ttf'
manjari_regular_path = 'Manjari-Regular.ttf'
manjari_thin_path = 'Manjari-thin.ttf'

dark = "#3B3365"
medium = "#8F81DD"
light = "#BCADFF"
peach = "#FFEEDD"

# Load the custom font
manjari_bold = fm.FontProperties(fname=manjari_bold_path)
manjari_regular = fm.FontProperties(fname=manjari_regular_path)
manjari_thin = fm.FontProperties(fname=manjari_thin_path)
df

,staff,replaced,Date,action_time,order_id,category,artist,machine,detail,quantity,base,total
0,Adam,NaN,2024-06-13,2024-06-13 16:16:45.000,TO24061339490416450897X86498,Photo,NaN,Blue Machine,Photo,1.0,10.0,10.0
1,Adam,NaN,2024-06-13,2024-06-13 16:21:03.000,TO24061341260421030546Z36498,Photo,NaN,Blue Machine,Photo,1.0,10.0,10.0
2,Adam,NaN,2024-06-13,2024-06-13 16:26:41.000,TO24061343680426400727ED6498,Photo,NaN,Blue Machine,Photo,1.0,10.0,10.0
3,Adam,NaN,2024-06-13,2024-06-13 16:32:58.000,TO240613462904325706909A6498,Photo,NaN,Blue Machine,Photo,1.0,10.0,10.0
4,Adam,NaN,2024-06-13,2024-06-13 16:39:26.000,TO24061348810439250204JN6498,Photo,NaN,Blue Machine,Photo,1.0,10.0,10.0
...,...,...,...,...,...,...,...,...,...,...,...,...
2508,Georgina,NaN,2024-08-16,2024-08-16 16:18:39.000,TO24081676580418390307N36497,Photo,NaN,Purple Machine,Copy,1.0,5.0,5.0
2509,Georgina,NaN,2024-08-16,2024-08-16 16:21:40.468,MmAboDEzFXp4T64XmE5yZNleV,Miscellaneous,NaN,NaN,NaN,1.0,5.0,5.0
2510,Georgina,NaN,2024-08-16,2024-08-16 16:22:21.000,TO24081679010422200597PO6497,Photo,NaN,Purple Machine,Photo,1.0,10.0,10.0
2511,Georgina,NaN,2024-08-16,2024-08-16 16:49:43.000,TO24081698740449420464O86497,Photo,NaN,Purple Machine,Photo,1.0,10.0,10.0


In [ ]:
for entry in range(len(data)):

  sample_df = pd.read_csv(data[entry]['input'])
  columns = str(list(sample_df.columns))
  data[entry]['columns'] = columns



In [ ]:
from nltk.translate.bleu_score import sentence_bleu

def compute_bleu(predictions, references, language='python'):
    """
    Computes the BLEU score.

    Args:
        predictions (list of str): Model-generated outputs.
        references (list of str): Ground truth outputs.
        language (str): Programming language of the code (e.g., "python", "java").

    Returns:
        dict: Dictionary containing the CodeBLEU score.
    """
    bleu_score = sentence_bleu([predictions], references)
    return {"codebleu": bleu_score}

def compute_metrics(eval_pred):
    """
    Wrapper for HuggingFace Trainer to compute CodeBLEU.

    Args:
        eval_pred: A tuple of (predictions, labels).

    Returns:
        dict: Dictionary of evaluation metrics.
    """
    predictions, labels = eval_pred

    # Convert model token IDs back to strings
    decoded_predictions = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Compute CodeBLEU
    return compute_bleu(decoded_predictions, decoded_labels, language="python")

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
# load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.



# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Meta-Llama-3.1-8B-bnb-4bit",      # Llama-3.1 2x faster
    "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Meta-Llama-3.1-70B-bnb-4bit",
    "unsloth/Meta-Llama-3.1-405B-bnb-4bit",    # 4bit for 405b!
    "unsloth/Mistral-Small-Instruct-2409",     # Mistral 22b 2x faster!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3.5-mini-instruct",           # Phi-3.5 2x faster!
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",            # Gemma 2x faster!

    "unsloth/Llama-3.2-1B-bnb-4bit",           # NEW! Llama 3.2 models
    "unsloth/Llama-3.2-1B-Instruct-bnb-4bit",
    "unsloth/Llama-3.2-3B-bnb-4bit",
    "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
] # More models at https://huggingface.co/unsloth



alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""



dataset = Dataset.from_dict({
      'instruction': [entry['instruction'].lower() for entry in data],
      'input': [entry['columns'] for entry in data],
      'output': [entry['output'] for entry in data]
  })


#80 training, #10 validation, #10 testing

split_dataset = dataset.train_test_split(test_size=0.2, seed=89)

validation_test_split = split_dataset['test'].train_test_split(test_size=0.5, seed=42)

train = split_dataset['train']
val = validation_test_split['train']
test = validation_test_split['test']




🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


In [ ]:
from trl import SFTTrainer
from unsloth import is_bfloat16_supported
from transformers import TrainingArguments, EarlyStoppingCallback, DataCollatorForLanguageModeling


#parameters to test out
models_to_test = [
    "unsloth/Llama-3.2-3B-Instruct",
    "unsloth/mistral-7b-instruct-v0.3",

]

learning_rates = [2e-4,1e-3]
lora_alphas = [16,32]
use_4bit_list = [True]


all_models = []

callbacks = [
    EarlyStoppingCallback(early_stopping_patience=4), #early stopping
]

def make_name(model_name,lr,use_4bit, lora_alpha):
  if use_4bit:
    return f"{model_name.replace('unsloth/','')}_{lr}_{lora_alpha}_4bit_quantized"
  return f"{model_name.replace('unsloth/','')}_{lr}_{lora_alpha}"

model_name = "unsloth/Llama-3.2-3B-Instruct"
lr = 2e-4
use_4bit = True
lora_alpha = 16


model_filename = make_name(model_name, lr, use_4bit, lora_alpha)
all_models.append(model_filename)
print(f'Now training {model_filename}')
model, tokenizer = FastLanguageModel.from_pretrained(
  model_name = model_name,
  max_seq_length = max_seq_length,
  dtype = dtype,
  load_in_4bit = use_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = lora_alpha,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

#dataset stuff

EOS_TOKEN = tokenizer.eos_token # Must add EOS_TOKEN

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        # Must add EOS_TOKEN, otherwise your generation will go on forever!
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)

    return { "text" : texts, }


train = train.map(formatting_prompts_func, batched = True,)
val = val.map(formatting_prompts_func, batched = True,)
test = test.map(formatting_prompts_func, batched = True,)

trainer = SFTTrainer(
  model = model,
  tokenizer = tokenizer,
  train_dataset = train,
  eval_dataset = val,
  dataset_text_field = "text",
  # data_collator=data_collator,
  max_seq_length = max_seq_length,
  dataset_num_proc = 2,
  packing = False, # Can make training 5x faster for short sequences.
  # compute_metrics=compute_metrics,
  callbacks=callbacks,
  args = TrainingArguments(
      per_device_train_batch_size = 2,
      gradient_accumulation_steps = 4,
      warmup_steps = 5,
      # num_train_epochs = 1000, # Set this for 1 full training run.
      max_steps = 200,
      learning_rate = lr,
      fp16 = not is_bfloat16_supported(),
      bf16 = is_bfloat16_supported(),
      optim = "adamw_8bit",
      weight_decay = 0.01,
      lr_scheduler_type = "linear",
      seed = 3407,
      evaluation_strategy="steps",
      logging_strategy="steps",
      save_strategy="steps",
      eval_steps = 25,
      save_steps = 25,
      logging_steps = 1,
      output_dir = "outputs",
      report_to = "wandb",
      save_total_limit = 3,
      run_name = model_filename,
      load_best_model_at_end = True,
  ),

)

run = wandb.init(project="ChoAI", name=model_filename)


# try:
  # artifact = run.use_artifact(f'kasmello/ChoAI/{run.id}', type='model')
  # print('Artifact found!')
  # artifact_dir = artifact.download()
try:
  trainer.train(resume_from_checkpoint=False)
except ValueError:
  print('uhoh')
  pass
# except wandb.CommError:
#   print('Artifact not found - training from scratch')
#   trainer.train()
wandb.log({
    'model_name': model_name.replace('unsloth/',''),
    'lr': lr,
    'use_4bit': use_4bit,
    'lora_alpha': lora_alpha
})
model.save_pretrained(model_filename) # Local saving
tokenizer.save_pretrained(model_filename)
# wandb.finish()

Now training Llama-3.2-3B-Instruct_0.0002_16_4bit_quantized
==((====))==  Unsloth 2024.11.7: Fast Llama patching. Transformers = 4.46.2.
   \\   /|    GPU: Tesla T4. Max memory: 14.748 GB. Platform = Linux.
O^O/ \_/ \    Pytorch: 2.5.1+cu121. CUDA = 7.5. CUDA Toolkit = 12.1.
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.28.post3. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:  60%|######    | 1.35G/2.24G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Unsloth 2024.11.7 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Map:   0%|          | 0/1432 [00:00<?, ? examples/s]

...

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs = 1
   \\   /|    Num examples = 1,432 | Num Epochs = 2
O^O/ \_/ \    Batch size per device = 2 | Gradient Accumulation steps = 4
\        /    Total batch size = 8 | Total steps = 200
 "-____-"     Number of trainable parameters = 24,313,856


Step,Training Loss,Validation Loss
25,0.370200,0.353292
50,0.168500,0.207573
75,0.211400,0.149209
100,0.149800,0.124544
125,0.091600,0.102491
150,0.078700,0.092042
175,0.083100,0.083360
200,0.071900,0.079835


wandb: Adding directory to artifact (./outputs/checkpoint-25)... Done. 0.4s
wandb: Adding directory to artifact (./outputs/checkpoint-50)... Done. 0.4s
wandb: Adding directory to artifact (./outputs/checkpoint-75)... Done. 0.4s
wandb: Adding directory to artifact (./outputs/checkpoint-100)... Done. 0.4s
wandb: Adding directory to artifact (./outputs/checkpoint-125)... Done. 0.4s
wandb: Adding directory to artifact (./outputs/checkpoint-150)... Done. 0.4s
wandb: Adding directory to artifact (./outputs/checkpoint-175)... Done. 0.4s
wandb: Adding directory to artifact (./outputs/checkpoint-200)... Done. 0.4s


uhoh


('Llama-3.2-3B-Instruct_0.0002_16_4bit_quantized/tokenizer_config.json',
 'Llama-3.2-3B-Instruct_0.0002_16_4bit_quantized/special_tokens_map.json',
 'Llama-3.2-3B-Instruct_0.0002_16_4bit_quantized/tokenizer.json')

In [ ]:


trainer_EVAL = SFTTrainer(
  model = model,
  tokenizer = tokenizer,
  eval_dataset = test,
  dataset_text_field = "text",
  # data_collator=data_collator,
  max_seq_length = max_seq_length,
  dataset_num_proc = 2,
  packing = False, # Can make training 5x faster for short sequences.
  callbacks=callbacks,
  args = TrainingArguments(

      seed = 3407,
      evaluation_strategy="steps",
      logging_strategy="steps",
      save_strategy="steps",
      eval_steps = 10,
      save_steps = 10,
      logging_steps = 1,
      output_dir = "results",
      report_to = "wandb",
      save_total_limit = 3,
      run_name = model_filename,
      load_best_model_at_end = True,
      do_eval = True
  ),

)


# try:
  # artifact = run.use_artifact(f'kasmello/ChoAI/{run.id}', type='model')
  # print('Artifact found!')
  # artifact_dir = artifact.download()
results = trainer_EVAL.evaluate()
wandb.log({"Testing Evaluation Loss": results['eval_loss']})
# except wandb.CommError:
#   print('Artifact not found - training from scratch')
#   trainer.train()
# wandb.log({
#     'model_name': model_name.replace('unsloth/',''),
#     'lr': lr,
#     'use_4bit': use_4bit,
#     'lora_alpha': lora_alpha
# })
# model.save_pretrained(model_filename) # Local saving
# tokenizer.save_pretrained(model_filename)
# wandb.finish()

Map (num_proc=2):   0%|          | 0/179 [00:00<?, ? examples/s]

In [ ]:
trainer_EVAL

{'eval_loss': 0.0770912617444992,
 'eval_model_preparation_time': 0.0089,
 'eval_runtime': 31.0676,
 'eval_samples_per_second': 5.762,
 'eval_steps_per_second': 0.74}

In [ ]:
import torch
import gc
torch.cuda.empty_cache()
del trainer
gc.collect()


9085